# Column Transformer

## 1. Connection with Feature Transformation

The previous section established that preprocessing may involve several kinds of transformations:

- **Numerical columns** → scaling / numerical transformation
- **Ordinal categorical columns** → ordinal encoding
- **Nominal categorical columns** → one-hot encoding
- **Output/target categorical column** → label encoding

When a dataset contains different column types, applying the correct transformation to each column separately can become repetitive.

**Column Transformer** provides a way to apply different transformations to different columns while keeping the preprocessing organized.

---

## 2. Why Column Transformer?

Suppose a dataset contains:

```text
Age        → numerical
Salary     → numerical
Education  → ordinal categorical
State      → nominal categorical
Target     → categorical output
```

We may need different operations:

```text
Age, Salary       → numerical transformation
Education         → OrdinalEncoder
State             → OneHotEncoder
Target            → LabelEncoder
```

The notes emphasize the idea of working with **independent features** without repeatedly separating columns from one another.

> **Key idea:** A Column Transformer lets us define which transformation should be applied to which columns.


## 3. Column Transformer — Basic Idea

The handwritten notes show the progression:

```text
Independent features
        ↓
Columns need different transformations
        ↓
Column Transformer
        ↓
Apply the selected transformer to selected columns
```

Instead of manually separating the columns every time, we can specify the transformations together.

### Main advantages

1. Select particular columns.
2. Assign a transformer to each selected group.
3. Apply different preprocessing operations in one pipeline-like structure.
4. Keep the preprocessing steps organized.


## 4. Transformations Mentioned in the Notes

The notes connect Column Transformer with these common scikit-learn transformers:

| Data / purpose | Transformer |
|---|---|
| Ordinal categorical data | `OrdinalEncoder` |
| Nominal categorical data | `OneHotEncoder` |
| Missing values | `SimpleImputer` |
| Numerical transformation | Scaling / other numerical transformer |

The important distinction is that **the transformer depends on the type and purpose of the column**.


## 5. `ColumnTransformer` Structure

The basic structure is:

```text
ColumnTransformer
        │
        ├── Transformer 1 → Columns A, B
        │
        ├── Transformer 2 → Columns C
        │
        └── Transformer 3 → Columns D, E
```

A transformer specification is conceptually written as:

```text
(name, transformer, columns)
```

### Meaning of the three parameters

1. **Name**  
   A name given to that transformation step.

2. **Transformer**  
   The preprocessing operation to perform, such as:
   - `OrdinalEncoder()`
   - `OneHotEncoder()`
   - `SimpleImputer()`

3. **Columns**  
   The columns on which that transformer should operate.

The notes specifically raise the question:

> Which columns should each transformation be applied to?

This is the role of the **columns** argument.


## 6. Parameter Format

A Column Transformer is commonly defined using a list of transformer specifications:

```python
ColumnTransformer(
    transformers=[
        ("name", transformer, columns),
        ("name", transformer, columns),
    ]
)
```

For example:

```python
("education", OrdinalEncoder(), ["education"])
("state", OneHotEncoder(), ["state"])
```

Here:

- `"education"` is the transformer name.
- `OrdinalEncoder()` is the transformer.
- `["education"]` identifies the column.

The notes also indicate that columns can be identified by **column name** or by **column position/index**.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

preprocessor = ColumnTransformer(
    transformers=[
        ("education", OrdinalEncoder(), ["education"]),
        ("state", OneHotEncoder(handle_unknown="ignore"), ["state"]),
        ("age_imputer", SimpleImputer(strategy="median"), ["age"]),
    ]
)

preprocessor


## 7. Example Dataset

A simple dataset can contain several different feature types:

| age | education | state |
|---:|---|---|
| 23 | BTech | Maharashtra |
| 35 | MTech | Delhi |
| 28 | PhD | Telangana |

Possible preprocessing:

```text
age
 ↓
SimpleImputer / numerical transformation

education
 ↓
OrdinalEncoder

state
 ↓
OneHotEncoder
```

This is the central use case for `ColumnTransformer`: **different columns can receive different transformations in a single preprocessing object.**


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "age": [23, 35, 28, 40],
    "education": ["BTech", "MTech", "BTech", "PhD"],
    "state": ["Maharashtra", "Delhi", "Telangana", "Delhi"]
})

df


## 8. Applying Different Transformers to Different Columns

For the education column, the categories have an order:

```text
BTech < MTech < PhD
```

So an **ordinal encoding** approach is appropriate.

For state, there is no meaningful order:

```text
Delhi
Maharashtra
Telangana
```

So a **one-hot encoding** approach is appropriate.

This gives:

```text
Education → OrdinalEncoder
State     → OneHotEncoder
```


In [ ]:
education_order = [["BTech", "MTech", "PhD"]]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "education",
            OrdinalEncoder(categories=education_order),
            ["education"]
        ),
        (
            "state",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["state"]
        )
    ]
)

transformed = preprocessor.fit_transform(df)
transformed


## 9. Column Selection

The notes emphasize that the user must decide **which columns** each transformer should work on.

Columns can be selected using:

### Column names

```python
["education", "state"]
```

### Column indexes

For example:

```python
[1]
```

or:

```python
[1, 2]
```

The exact choice depends on how the dataset is represented.

> **Important:** The column selection determines where each transformation is applied.


## 10. Handling Columns That Are Not Explicitly Listed

The notes also discuss:

```text
drop
passthrough
```

These relate to the `remainder` behavior of `ColumnTransformer`.

If some columns are not included in the transformer specifications, `remainder` determines what happens to those columns.

### `remainder="drop"`

Columns not selected for a transformation are removed.

```python
ColumnTransformer(
    transformers=[...],
    remainder="drop"
)
```

### `remainder="passthrough"`

Columns not selected for a transformation are kept without applying a transformer to them.

```python
ColumnTransformer(
    transformers=[...],
    remainder="passthrough"
)
```

The notes point out that this behavior applies to the **columns left out of the explicitly specified transformations**.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Columns not mentioned in transformers are dropped.
ct_drop = ColumnTransformer(
    transformers=[
        ("state", OneHotEncoder(handle_unknown="ignore"), ["state"])
    ],
    remainder="drop"
)

# Columns not mentioned in transformers are passed through unchanged.
ct_pass = ColumnTransformer(
    transformers=[
        ("state", OneHotEncoder(handle_unknown="ignore"), ["state"])
    ],
    remainder="passthrough"
)


## 11. Transformation on Multiple Columns

The notes give the idea that the same transformer can be applied to a group of columns.

For example:

```python
("numeric", transformer, ["age", "salary"])
```

means that the selected transformer is applied to both `age` and `salary`.

Conceptually:

```text
                 ┌── age
Numeric transformer
                 └── salary

Ordinal transformer
                 └── education

One-hot transformer
                 └── state
```

This avoids repeatedly writing separate preprocessing operations for every individual column when the same operation is required.


## 12. Complete Example

The following example combines several ideas from the notes:

- Numerical missing-value handling
- Ordinal encoding
- Nominal one-hot encoding
- Keeping selected columns organized through `ColumnTransformer`


In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

df = pd.DataFrame({
    "age": [23, None, 28, 40],
    "education": ["BTech", "MTech", "BTech", "PhD"],
    "state": ["Maharashtra", "Delhi", "Telangana", "Delhi"]
})

education_order = [["BTech", "MTech", "PhD"]]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "age_imputer",
            SimpleImputer(strategy="median"),
            ["age"]
        ),
        (
            "education",
            OrdinalEncoder(categories=education_order),
            ["education"]
        ),
        (
            "state",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["state"]
        )
    ],
    remainder="drop"
)

X_transformed = preprocessor.fit_transform(df)

X_transformed


## 13. Important Relationship: Encoding + Column Transformer

The notes establish a useful mapping:

```text
Categorical data
      │
      ├── Nominal → OneHotEncoder
      │
      └── Ordinal → OrdinalEncoder
```

And:

```text
Multiple feature types
        ↓
ColumnTransformer
        ↓
Different transformer for each relevant column/group
```

Therefore, `ColumnTransformer` is not itself an encoding technique.

It is a **framework for organizing and applying different transformations to selected columns**.


## 14. Where `SimpleImputer` Fits

The notes include `SimpleImputer` alongside the encoders as a transformer that can be supplied to the Column Transformer.

For example:

```python
("age", SimpleImputer(strategy="median"), ["age"])
```

The idea is:

```text
Missing numerical values
        ↓
SimpleImputer
        ↓
Completed numerical column
```

The imputer can be assigned only to the columns where missing-value handling is required.


## 15. Study Diagram

```text
                    DATASET
                       │
          ┌────────────┼────────────┐
          │            │            │
       Numerical     Ordinal      Nominal
        columns      columns      columns
          │            │            │
          ↓            ↓            ↓
    Imputer /      Ordinal       One-Hot
    Scaler         Encoder       Encoder
          │            │            │
          └────────────┼────────────┘
                       ↓
                ColumnTransformer
                       ↓
             Transformed feature matrix
```

This diagram connects the separate transformations into the single preprocessing structure discussed in the notes.


## 16. Important Points

- **ColumnTransformer** is used when different columns require different transformations.
- A transformer specification follows the idea:

  ```text
  (name, transformer, columns)
  ```

- `OrdinalEncoder` is used for ordered categorical features.
- `OneHotEncoder` is used for nominal categorical features.
- `SimpleImputer` can be used for missing-value handling.
- Columns can be selected by **name** or **index**.
- Multiple columns can be supplied to one transformer when they need the same operation.
- `remainder="drop"` removes columns that were not explicitly transformed.
- `remainder="passthrough"` keeps columns that were not explicitly transformed.
- Column Transformer helps avoid repeatedly separating and preprocessing columns manually.


## 17. Quick Revision

### One-line definition

**Column Transformer allows different preprocessing transformations to be applied to different columns of the same dataset.**

### Remember the pattern

```text
ColumnTransformer
    ↓
(name, transformer, columns)
```

### Common mapping

| Requirement | Transformer |
|---|---|
| Ordered categorical feature | `OrdinalEncoder` |
| Unordered categorical feature | `OneHotEncoder` |
| Missing values | `SimpleImputer` |
| Unselected columns | `remainder="drop"` / `remainder="passthrough"` |

### Final mental model

```text
Different column types
        ↓
Different transformations
        ↓
ColumnTransformer
        ↓
One organized preprocessing step


## 18. Additional Explanation

A practical machine-learning preprocessing workflow often looks like:

```text
Raw Data
   ↓
Identify column types
   ↓
Choose transformation for each type
   ↓
ColumnTransformer
   ↓
Transformed features
   ↓
Machine-learning model
```

The main benefit is consistency: the same preprocessing definition can later be reused when transforming validation or test data, instead of manually repeating the column-by-column operations.
